# Appendix — OpenAI Agents SDK: the same triage, gated

The [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/) is, in its own words, "a lightweight, easy-to-use package with very few abstractions." This appendix runs the course's fixed ticket — **TKT-2205** — through it: same tools, same world data, same economics you have hit in every other appendix. The point is not to learn a new agent idea. It is to see the `Agent` / `Runner` loop, the `@function_tool` decorator, and the SDK's built-in approval gate line up, one for one, against the machinery you built by hand in Parts 1-3.

> **Before running this notebook** — the OpenAI Agents SDK installs into its **own** virtual environment with its **own** Jupyter kernel. It needs `openai>=3`, which conflicts with the `openai<3` that `litellm` pins in the main stack (resolver-proven), so it cannot share the course venv. Set it up once:
>
> ```
> python -m venv .venv-openai-agents && . .venv-openai-agents/bin/activate
> pip install -e ".[openai-agents]"
> python -m ipykernel install --user --name agentic-lab-openai-agents
> ```
>
> Then, in Jupyter, pick the kernel **agentic-lab-openai-agents** for THIS notebook (Kernel -> Change Kernel). Every cell below runs in that venv — `agents`, plus the same `shoplab`, `litellm`, and `diskcache` you already know.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

## The invariant: what TKT-2205 must decide

Every framework appendix ends at the same numbers so they are comparable. The gold is not hand-typed — it comes straight from `shoplab.rules.decide`, the deterministic cascade the whole world is graded against. For TKT-2205 (an *opened* boot returned by a *non-vip member*, 18 days out, inside the 30-day window) branch 9 fires: item value minus a 10% restocking fee.

In [ ]:
# The one fixed decision this appendix must land on (authoritative).
from shoplab import world
from shoplab.rules import decide

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}
ticket = next(t for t in world.load_tickets()["train"] if t["ticket_id"] == "TKT-2205")
order, customer = orders[ticket["order_id"]], customers[ticket["customer_id"]]

GOLD = decide(ticket, order, customer)
print("TKT-2205 gold:", GOLD)          # partial_refund / pol-restocking / 170.99
print("arithmetic:", 189.99, "x 0.90 =", round(189.99 * 0.90, 2))

## Wiring the course model into the SDK

The SDK talks to OpenAI by default, but ships a LiteLLM adapter — `LitellmModel` — that takes the same provider-prefixed model string you have used since ch01. So `openrouter/deepseek/deepseek-v3.2` drops straight in, and because the call still goes through `litellm`, the disk cache from the config cell makes reruns ~free. One extra line: `set_tracing_disabled(True)`, because the SDK's default trace exporter wants an OpenAI platform key we are not using.

In [ ]:
from agents import Agent, Runner, function_tool, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

set_tracing_disabled(True)   # the default trace exporter wants an OpenAI key; we don't use it
model = LitellmModel(model=MODEL, api_key=os.environ["OPENROUTER_API_KEY"])
print("model:", MODEL)

## The four ops-desk tools, as `@function_tool`

`@function_tool` turns a plain function's signature and docstring into a tool schema — exactly the job `to_openai_tools` did by hand in ch02. The wrappers stay thin: each one calls straight into the `shoplab` tool it stands for, so the world data and the risky-tool `Ledger` are the same objects the rest of the course uses. The one that moves money, `issue_refund`, carries `needs_approval=True` — the SDK's native human-in-the-loop gate, declared right on the tool.

In [ ]:
import json
from shoplab.tools import standard_tools, Ledger

ledger = Ledger()            # ch02's risky-tool audit log
_t = standard_tools(ledger)  # the 9 course tools, backed by the world data

@function_tool
def get_order(order_id: str) -> str:
    """Look up an order by id: items, totals, status, dates."""
    return json.dumps(_t["get_order"].fn(order_id))

@function_tool
def search_policy(query: str, k: int = 2) -> str:
    """Keyword-search the 12 store policy documents."""
    return json.dumps(world.search_policy(query, k))

@function_tool
def calc(expr: str) -> str:
    """Evaluate an arithmetic expression, e.g. '0.9 * 189.99'."""
    return json.dumps(_t["calc"].fn(expr))

In [ ]:
@function_tool(needs_approval=True)   # the ch08 approval gate, declared on the tool
def issue_refund(order_id: str, amount_usd: float, reason: str) -> str:
    """Send money back to the customer. Irreversible."""
    return json.dumps(_t["issue_refund"].fn(order_id, amount_usd, reason))

TOOLS = [get_order, search_policy, calc, issue_refund]

## Run the triage — the gate holds the refund

`Runner.run` is the SDK's loop: model call, tool calls, repeat, until the agent answers. When the agent reaches `issue_refund`, `needs_approval` stops the run and hands back an *interruption* — a pending tool call awaiting a human — instead of moving money. `result.interruptions` is that queue; the `Ledger` is still empty, because nothing has executed yet.

In [ ]:
SYSTEM = (
    "You are the Larkspur Outfitters ops-desk agent. Triage one return ticket end to end "
    "with your tools, in order: get_order for the unit price, search_policy for the governing "
    "policy, calc the refund (item value minus a 10% restocking fee, rounded to the nearest "
    "cent), then issue_refund for that exact amount. You MUST call issue_refund and see its "
    "result before you answer. Then state the decision, the policy id, and the dollar amount."
)
TASK = (
    "Ticket TKT-2205: order ORD-7312, customer CUST-07 (member, non-vip), sku LK-1016 qty 1, "
    "condition opened, 18 days since delivery, requests a refund to the original payment method."
)
agent = Agent(name="ops-desk", instructions=SYSTEM, model=model, tools=TOOLS)

result = await Runner.run(agent, TASK, max_turns=10)
for gate in result.interruptions:
    print("APPROVAL NEEDED ->", gate.tool_name, getattr(gate.raw_item, "arguments", None))
print("ledger (has the refund executed?):", ledger.entries)

> **What you should see:** one approval interruption for `issue_refund`, its arguments already carrying `amount_usd` = **170.99**, and the ledger **still empty**. The gate paused the loop the instant the agent reached for money — before any of it moved.

## Approve, resume, and land

A human approves the pending call on the run's *state* (`state.reject(gate)` would block it and let the agent recover), then `Runner.run` resumes from exactly where it stopped. Now `issue_refund` executes, the `Ledger` records it, and the agent finishes. We check the money it actually moved against the gold.

In [ ]:
state = result.to_state()
for gate in result.interruptions:
    state.approve(gate)                 # a human says yes

result = await Runner.run(agent, state, max_turns=10)
print("final answer:", result.final_output)
print("ledger:", ledger.entries)

moved = ledger.entries[-1]["amount_usd"]
print(f"agent moved ${moved}  |  gold ${GOLD['refund_usd']}  |  match: {moved == GOLD['refund_usd']}")

> **What you should see:** after approval the refund executes; the ledger holds a single `issue_refund` entry for **$170.99**, matching the gold exactly — the same **partial_refund / pol-restocking / 170.99** every appendix lands on. The final answer names that decision in prose (the SDK's typed `output_type` was left off on purpose: with this model it tempts the agent to skip the gated tool and guess the number, which defeats the whole demo).

## Machinery map: SDK concept to the part you built

Line them up and the SDK stops being magic — it is Parts 1-3, packaged with defaults and a runtime. Nothing in this appendix is a new idea about agents; it is the ops-desk loop you already wrote, wearing the SDK's names.

| OpenAI Agents SDK | Your hand-built equivalent | Built in |
|---|---|---|
| `Agent` + `Runner.run` | `run_agent`: model call -> execute tool calls -> repeat until done | ch02 |
| `@function_tool` (signature + docstring -> schema) | `to_openai_tools` over the `Tool` registry | ch02 |
| `LitellmModel(model=MODEL)` | `shoplab.llm.complete` over LiteLLM, same provider string | ch01 |
| `result.new_items` / `to_state()` | the message list you threaded through the loop by hand | ch02 |
| a tool returning an `{"error": ...}` string the model reads | `run_tool` swallowing exceptions into an observation | ch02 |
| `needs_approval=True` + `state.approve()` | `require_approval` wrapping a risky `Tool` in a gate | ch08 |
| `input_guardrail` / `handoff` (shipped, unused here) | ch08's tripwire check / ch07's handoff to another agent | ch07/08 |

## The honest read

The SDK gave us three things for free that we built by hand: the loop (`Runner`), the schema plumbing (`@function_tool`), and a real approval gate (`needs_approval` + interruptions). What it did not change is the hard part — the world, the tools, the rules, and the decision. Those are still `shoplab`. An appendix, not a rewrite: the framework is a convenience over machinery you now understand well enough to have skipped it.